# Venous Cell Zoom Across Models

Loads saved evaluated AnnData outputs from the human/monkey/mouse walkthrough and draws a static venous-cell UMAP panel for all four models.

Uses the JAX backend and native Orbax checkpoints. Install the appropriate `jax_saturn/requirements*.txt` dependencies first. Saved outputs must be regenerated for this backend; this notebook has not been executed as part of the port.


In [ ]:
from pathlib import Path
import os
import re

import anndata as ad
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = next((
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'scripts' / 'human_monkey_mouse_jax.sh').exists()
), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate the SATURN repository root.')

# Override this with HMM_REPORT_SOURCE_DIR if you want a different completed run.
figure_source_dir = Path(os.environ.get(
    'HMM_REPORT_SOURCE_DIR',
    repo_root / 'out' / 'human_monkey_mouse_jax_benchmark_walkthrough',
)).expanduser()
if not figure_source_dir.is_absolute():
    figure_source_dir = repo_root / figure_source_dir
figure_source_dir = figure_source_dir.resolve()

HIGHLIGHT_FINE_LABEL = 'venous'
MATCH_MODE = 'exact'
SAVE_FIGURE = False

TRIALS = ['baseline', 'infonce', 'mmd', 'ot']
TRIAL_DISPLAY = {
    'baseline': 'SATURN baseline',
    'infonce': 'InfoNCE',
    'mmd': 'MMD-MA',
    'ot': 'Partial OT',
}
SPECIES_ORDER = ['h_sapiens', 'm_murinus', 'm_musculus']
SPECIES_DISPLAY = {
    'h_sapiens': 'human',
    'm_murinus': 'murinus',
    'm_musculus': 'musculus',
}
SPECIES_PALETTE = {
    'h_sapiens': '#1f77b4',
    'm_murinus': '#ff7f0e',
    'm_musculus': '#2ca02c',
}

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

query = str(HIGHLIGHT_FINE_LABEL).strip()
if not query:
    raise ValueError('Set HIGHLIGHT_FINE_LABEL to a non-empty labels2 value or substring.')
query_norm = query.casefold()

def fine_label_mask(labels, mode):
    values = pd.Series(labels).astype(str)
    normalized = values.str.casefold()
    if mode == 'contains':
        return normalized.str.contains(re.escape(query_norm), na=False).to_numpy()
    if mode == 'exact':
        return (normalized == query_norm).to_numpy()
    raise ValueError("MATCH_MODE must be either 'contains' or 'exact'.")

def load_trial_frame(trial):
    path = figure_source_dir / trial / 'evaluated_adata.h5ad'
    if not path.exists():
        raise FileNotFoundError(
            f'Missing evaluated AnnData for {trial}: {path}. '
            'Run the benchmark once or set HMM_REPORT_SOURCE_DIR to a completed run.'
        )
    trial_adata = ad.read_h5ad(path, backed='r')
    try:
        if 'X_umap' not in trial_adata.obsm:
            raise KeyError(f'{path} does not contain obsm["X_umap"].')
        umap = np.asarray(trial_adata.obsm['X_umap'])
        frame = pd.DataFrame({
            'umap_1': umap[:, 0],
            'umap_2': umap[:, 1],
            'species': trial_adata.obs['species'].astype(str).to_numpy(),
            'cell_type': trial_adata.obs['labels2'].astype(str).to_numpy(),
        })
    finally:
        trial_adata.file.close()
    frame['highlight'] = fine_label_mask(frame['cell_type'], MATCH_MODE)
    return frame

trial_frames = {trial: load_trial_frame(trial) for trial in TRIALS}

if not any(frame['highlight'].any() for frame in trial_frames.values()):
    available_labels = sorted(trial_frames['baseline']['cell_type'].unique())
    suggestions = [label for label in available_labels if query_norm in label.casefold()][:20]
    raise ValueError(
        f'No labels2 values matched {query!r} with MATCH_MODE={MATCH_MODE!r}. '
        f'Containing-label examples: {suggestions}'
    )

count_rows = []
for trial, frame in trial_frames.items():
    for species in SPECIES_ORDER:
        count_rows.append({
            'method': TRIAL_DISPLAY[trial],
            'species': SPECIES_DISPLAY[species],
            'highlighted_cells': int((frame['highlight'] & (frame['species'] == species)).sum()),
        })

count_frame = pd.DataFrame(count_rows)
display(count_frame.pivot(index='species', columns='method', values='highlighted_cells'))

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=False)
axes = axes.ravel()

for ax, trial in zip(axes, TRIALS):
    frame = trial_frames[trial]
    background = frame[~frame['highlight']]
    ax.scatter(
        background['umap_1'],
        background['umap_2'],
        s=1.1,
        c='#D0D0D0',
        alpha=0.16,
        linewidths=0,
        rasterized=True,
    )

    for species in SPECIES_ORDER:
        species_frame = frame[frame['highlight'] & (frame['species'] == species)]
        if species_frame.empty:
            continue
        ax.scatter(
            species_frame['umap_1'],
            species_frame['umap_2'],
            s=8,
            c=SPECIES_PALETTE[species],
            alpha=0.9,
            linewidths=0,
            rasterized=True,
        )

    ax.set_title(f"{TRIAL_DISPLAY[trial]} (n={int(frame['highlight'].sum()):,})")
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.grid(False)

legend_handles = [
    mlines.Line2D(
        [],
        [],
        marker='o',
        linestyle='None',
        markerfacecolor=SPECIES_PALETTE[species],
        markeredgecolor='none',
        markersize=7,
        label=SPECIES_DISPLAY[species],
    )
    for species in SPECIES_ORDER
]
fig.legend(
    handles=legend_handles,
    title='Highlighted species',
    loc='lower center',
    ncol=len(legend_handles),
    frameon=False,
)
fig.suptitle(f"All models: labels2 {MATCH_MODE} {query!r}", fontsize=14)
fig.tight_layout(rect=[0, 0.07, 1, 0.95])

if SAVE_FIGURE:
    assets = figure_source_dir / 'report_assets'
    assets.mkdir(parents=True, exist_ok=True)
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', query).strip('_') or 'highlight'
    for suffix in ('png', 'pdf'):
        path = assets / f'all_models_{slug}_static.{suffix}'
        fig.savefig(path, dpi=300, bbox_inches='tight')
        print(f'Saved {path.relative_to(repo_root)}')

plt.show()
